# Notebook 02: Document & Text Metrics Visualization

**Purpose:** Analyze and visualize text-level and document-level quality metrics  
from the OCR source documents and the LangGraph extraction pipeline outputs.

## Sections
1. OCR quality distribution (page-level)
2. Chunk-level token distribution by modality
3. Fabrication rate by feature (bar chart)
4. Fabrication rate by prompt variant (comparative)
5. Verification confidence distribution
6. Retrieval score vs. verification confidence scatter
7. Self-consistency agreement analysis
8. Multi-prompt comparison heatmap
9. Feature-level accuracy vs fabrication rate
10. Case-level safety score distribution

## 0. Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
        r"\Documents\GitHub\llm_summarization_br_ca",
    )
)
DATA_PRIVATE_DIR = Path(
    os.getenv("DATA_PRIVATE_DIR", r"C:\Users\jamesr4\loc\data_private")
)
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"REPORTS_DIR: {REPORTS_DIR}")

## 1. Load Feature Outputs

In [ ]:
# Primary: load from experiments/runs/ parquet
# Fallback: load from data/processed/

def find_latest_run_parquet(runs_dir: Path) -> Path | None:
    parquets = sorted(runs_dir.rglob("feature_outputs.parquet"))
    return parquets[-1] if parquets else None

runs_dir = PROJECT_ROOT / "experiments" / "runs"
parquet_path = find_latest_run_parquet(runs_dir)

if parquet_path:
    print(f"Loading from: {parquet_path}")
    df = pd.read_parquet(parquet_path)
else:
    # Fallback: load from data/processed/
    fallback_csv = PROJECT_ROOT / "data" / "processed" / "comprehensive_enhanced_dataset_with_all_metrics.csv"
    if fallback_csv.exists():
        df = pd.read_csv(fallback_csv)
        print(f"Loaded fallback dataset: {fallback_csv.name} ({len(df)} rows)")
    else:
        print("No data found. Generating synthetic demo data for visualization.")
        rng = np.random.default_rng(42)
        features = [
            "feature_1_lesion_size", "feature_2_lesion_location",
            "feature_3_calcifications_asymmetry", "feature_5_extent",
            "feature_6_accurate_clip_placement", "feature_7_workup_recommendation",
            "feature_8_lymph_node", "feature_10_biopsy_method",
            "feature_11_invasive_component_size_pathology",
            "feature_12_histologic_diagnosis", "feature_13_receptor_status",
        ]
        prompts = ["P1_zero_shot", "P2_cot", "P3_rag_verify_v1"]
        n = 100
        df = pd.DataFrame({
            "case_id": [f"CASE_{i:03d}" for i in rng.integers(1, 30, n)],
            "feature_name": rng.choice(features, n),
            "prompt_id": rng.choice(prompts, n),
            "model_id": "claude-3-5-sonnet-20241022",
            "verdict": rng.choice(
                ["CORRECT", "FABRICATION", "OMISSION", "UNCERTAIN"],
                n, p=[0.72, 0.10, 0.09, 0.09],
            ),
            "confidence": rng.uniform(0.3, 1.0, n),
            "verification_confidence": rng.choice(
                [0.0, 0.8, 1.0], n, p=[0.12, 0.25, 0.63]
            ),
            "retrieval_attempts": rng.choice([1, 2], n, p=[0.75, 0.25]),
            "supported": rng.choice([True, False, None], n, p=[0.78, 0.12, 0.10]),
            "verification_method": rng.choice(
                ["rag_verification", "self_consistency_passed",
                 "self_consistency_failed", "skipped_no_value"],
                n, p=[0.70, 0.12, 0.06, 0.12],
            ),
        })
        df["confidence"] = df["confidence"].round(3)

print(f"Dataset shape: {df.shape}")
df.head()

## 2. OCR Quality — Page-Level (if available)

In [ ]:
ocr_quality_path = PROJECT_ROOT / "data" / "features" / "page_level_ocr_quality.csv"

if ocr_quality_path.exists():
    ocr_df = pd.read_csv(ocr_quality_path)
    print(f"OCR quality data: {ocr_df.shape}")
    print(ocr_df.describe())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # OCR confidence distribution
    if "ocr_confidence" in ocr_df.columns:
        axes[0].hist(ocr_df["ocr_confidence"].dropna(), bins=20, color="#3498db", edgecolor="white")
        axes[0].set_title("OCR Confidence Distribution (Page-Level)", fontweight="bold")
        axes[0].set_xlabel("OCR Confidence Score")
        axes[0].set_ylabel("Page Count")
        axes[0].axvline(ocr_df["ocr_confidence"].median(), color="red",
                        linestyle="--", label=f"Median={ocr_df['ocr_confidence'].median():.2f}")
        axes[0].legend()

    # Word count per page
    if "word_count" in ocr_df.columns:
        axes[1].hist(ocr_df["word_count"].dropna(), bins=20, color="#9b59b6", edgecolor="white")
        axes[1].set_title("Word Count per Page", fontweight="bold")
        axes[1].set_xlabel("Word Count")
        axes[1].set_ylabel("Page Count")

    plt.suptitle("OCR Quality Metrics", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "ocr_quality_metrics.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(f"OCR quality file not found: {ocr_quality_path}")
    print("Skipping OCR quality section.")

## 3. Fabrication Rate by Feature

In [ ]:
from eval.metrics.fabrication_metrics import fabrication_by_feature

fab_by_feat = fabrication_by_feature(df)
fab_by_feat["feature_label"] = (
    fab_by_feat["feature_name"]
    .str.replace("feature_", "")
    .str.replace("_", " ")
    .str.title()
)

fig, ax = plt.subplots(figsize=(12, 6))

colors_fab = [
    "#e74c3c" if r >= 0.10 else "#f39c12" if r >= 0.05 else "#2ecc71"
    for r in fab_by_feat["fabrication_rate"]
]

bars = ax.barh(
    fab_by_feat["feature_label"],
    fab_by_feat["fabrication_rate"] * 100,
    color=colors_fab,
    edgecolor="white",
    linewidth=1.2,
)

ax.axvline(x=5, color="black", linestyle="--", alpha=0.5, linewidth=1.5, label="5% safety threshold")
ax.set_xlabel("Fabrication Rate (%)")
ax.set_title(
    "Fabrication Rate by Clinical Feature\n(Red > 10%, Orange 5-10%, Green < 5%)",
    fontweight="bold", fontsize=13,
)
ax.legend()

for bar, rate in zip(bars, fab_by_feat["fabrication_rate"]):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1%}",
        va="center", fontsize=9,
    )

plt.tight_layout()
plt.savefig(REPORTS_DIR / "fabrication_rate_by_feature.png", dpi=150, bbox_inches="tight")
plt.show()
fab_by_feat[["feature_label", "n", "fabrication_count", "fabrication_rate"]]

## 4. Fabrication Rate by Prompt Variant

In [ ]:
from eval.metrics.fabrication_metrics import fabrication_by_prompt

if "prompt_id" in df.columns:
    fab_by_prompt = fabrication_by_prompt(df)

    fig, ax = plt.subplots(figsize=(9, 5))

    x = range(len(fab_by_prompt))
    width = 0.35

    bars1 = ax.bar(
        [i - width / 2 for i in x],
        fab_by_prompt["fabrication_rate"] * 100,
        width=width,
        color="#e74c3c",
        label="Fabrication Rate",
        edgecolor="white",
    )
    bars2 = ax.bar(
        [i + width / 2 for i in x],
        fab_by_prompt["accuracy"] * 100,
        width=width,
        color="#2ecc71",
        label="Accuracy",
        edgecolor="white",
    )

    ax.set_xticks(list(x))
    ax.set_xticklabels(fab_by_prompt["prompt_id"], rotation=15)
    ax.set_ylabel("Rate (%)")
    ax.set_title("Fabrication Rate vs Accuracy by Prompt Variant", fontweight="bold", fontsize=13)
    ax.legend()
    ax.axhline(y=5, color="black", linestyle="--", alpha=0.4, label="5% safety threshold")

    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "fabrication_rate_by_prompt.png", dpi=150, bbox_inches="tight")
    plt.show()
    fab_by_prompt
else:
    print("No prompt_id column found — skipping.")

## 5. Verification Confidence Distribution

In [ ]:
vc_col = "verification_confidence"
if vc_col in df.columns:
    vc = df[vc_col].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Histogram
    axes[0].hist(vc, bins=20, color="#3498db", edgecolor="white", alpha=0.85)
    axes[0].axvline(0.8, color="red", linestyle="--", label="Pass threshold (0.8)")
    axes[0].axvline(vc.mean(), color="orange", linestyle="-.",
                    label=f"Mean={vc.mean():.2f}")
    axes[0].set_title("Verification Confidence Distribution", fontweight="bold")
    axes[0].set_xlabel("Verification Confidence")
    axes[0].set_ylabel("Count")
    axes[0].legend()

    # By verdict
    if "verdict" in df.columns:
        verdict_order = ["CORRECT", "FABRICATION", "OMISSION", "UNCERTAIN"]
        palette = {
            "CORRECT": "#2ecc71",
            "FABRICATION": "#e74c3c",
            "OMISSION": "#f39c12",
            "UNCERTAIN": "#95a5a6",
        }
        plot_df = df.dropna(subset=[vc_col, "verdict"])
        sns.boxplot(
            data=plot_df,
            x="verdict",
            y=vc_col,
            order=[v for v in verdict_order if v in plot_df["verdict"].unique()],
            palette=palette,
            ax=axes[1],
        )
        axes[1].set_title("Verification Confidence by Verdict", fontweight="bold")
        axes[1].set_xlabel("Verdict")
        axes[1].set_ylabel("Verification Confidence")
        axes[1].axhline(0.8, color="red", linestyle="--", alpha=0.6, label="Threshold")
        axes[1].legend()

    plt.suptitle("Verification Confidence Analysis", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "verification_confidence_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Verification pass rate (>=0.8): {(vc >= 0.8).mean():.1%}")
    print(f"Mean verification confidence: {vc.mean():.3f}")

## 6. Extraction Confidence vs. Verification Confidence Scatter

In [ ]:
if "confidence" in df.columns and "verification_confidence" in df.columns and "verdict" in df.columns:
    plot_df = df.dropna(subset=["confidence", "verification_confidence", "verdict"]).copy()

    palette = {
        "CORRECT": "#2ecc71",
        "FABRICATION": "#e74c3c",
        "OMISSION": "#f39c12",
        "UNCERTAIN": "#95a5a6",
    }

    fig, ax = plt.subplots(figsize=(9, 7))

    for verdict, group in plot_df.groupby("verdict"):
        ax.scatter(
            group["confidence"],
            group["verification_confidence"],
            label=verdict,
            color=palette.get(verdict, "#bdc3c7"),
            alpha=0.7,
            s=60,
            edgecolors="white",
            linewidths=0.5,
        )

    ax.axhline(y=0.8, color="red", linestyle="--", alpha=0.5, label="Verification threshold")
    ax.axvline(x=0.75, color="blue", linestyle="--", alpha=0.5, label="Extraction threshold")
    ax.set_xlabel("Extraction Confidence", fontsize=12)
    ax.set_ylabel("Verification Confidence", fontsize=12)
    ax.set_title(
        "Extraction vs. Verification Confidence\nColored by Final Verdict",
        fontweight="bold", fontsize=13,
    )
    ax.legend(title="Verdict", framealpha=0.9)
    ax.set_xlim(-0.05, 1.1)
    ax.set_ylim(-0.05, 1.1)

    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "confidence_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Multi-Prompt Fabrication Heatmap

In [ ]:
if "prompt_id" in df.columns and "feature_name" in df.columns and "verdict" in df.columns:
    pivot = df.groupby(["prompt_id", "feature_name"]).apply(
        lambda g: (g["verdict"] == "FABRICATION").mean()
    ).reset_index(name="fabrication_rate")

    heatmap_data = pivot.pivot(index="feature_name", columns="prompt_id", values="fabrication_rate")
    heatmap_data.index = [
        i.replace("feature_", "").replace("_", " ").title()
        for i in heatmap_data.index
    ]

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".1%",
        cmap="RdYlGn_r",
        vmin=0,
        vmax=0.25,
        ax=ax,
        linewidths=0.5,
        cbar_kws={"label": "Fabrication Rate"},
    )
    ax.set_title(
        "Fabrication Rate Heatmap: Feature × Prompt Variant",
        fontweight="bold", fontsize=13,
    )
    ax.set_xlabel("Prompt Variant")
    ax.set_ylabel("Clinical Feature")
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "fabrication_heatmap_feature_prompt.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Feature Accuracy vs. Fabrication Rate — Bubble Chart

In [ ]:
from eval.metrics.fabrication_metrics import fabrication_by_feature
from eval.metrics.extraction_metrics import compute_feature_metrics

fab_df = fabrication_by_feature(df)

if "verdict" in df.columns:
    acc_df = df.groupby("feature_name").apply(
        lambda g: (g["verdict"] == "CORRECT").mean()
    ).reset_index(name="accuracy")

    merged = fab_df.merge(acc_df, on="feature_name")
    merged["feature_label"] = (
        merged["feature_name"]
        .str.replace("feature_", "")
        .str.replace("_", " ")
        .str.title()
        .str[:20]
    )

    fig, ax = plt.subplots(figsize=(10, 7))

    sc = ax.scatter(
        merged["accuracy"] * 100,
        merged["fabrication_rate"] * 100,
        s=merged["n"] * 8,
        c=merged["fabrication_rate"],
        cmap="RdYlGn_r",
        alpha=0.8,
        edgecolors="white",
        linewidths=1.5,
        vmin=0, vmax=0.25,
    )

    for _, row in merged.iterrows():
        ax.annotate(
            row["feature_label"],
            xy=(row["accuracy"] * 100, row["fabrication_rate"] * 100),
            xytext=(5, 5), textcoords="offset points",
            fontsize=8, alpha=0.85,
        )

    ax.axhline(y=5, color="black", linestyle="--", alpha=0.4, label="5% safety threshold")
    plt.colorbar(sc, ax=ax, label="Fabrication Rate")
    ax.set_xlabel("Extraction Accuracy (%)")
    ax.set_ylabel("Fabrication Rate (%)")
    ax.set_title(
        "Feature Accuracy vs. Fabrication Rate\n(Bubble size = sample count)",
        fontweight="bold", fontsize=13,
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "accuracy_vs_fabrication_bubble.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9. Retrieval Attempts Distribution & Query Rewrite Rate

In [ ]:
from eval.metrics.retrieval_metrics import retrieval_summary_by_feature

if "retrieval_attempts" in df.columns:
    retrieval_summary = retrieval_summary_by_feature(df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Rewrite rate by feature
    retrieval_summary["feature_short"] = (
        retrieval_summary["feature_name"]
        .str.replace("feature_", "")
        .str.replace("_", " ")
        .str.title()
        .str[:18]
    )
    axes[0].barh(
        retrieval_summary["feature_short"],
        retrieval_summary["rewrite_rate"] * 100,
        color="#9b59b6",
        edgecolor="white",
    )
    axes[0].set_title("Query Rewrite Rate by Feature", fontweight="bold")
    axes[0].set_xlabel("Rewrite Rate (%)")

    # Distribution of retrieval attempts
    attempt_counts = df["retrieval_attempts"].value_counts().sort_index()
    axes[1].bar(
        attempt_counts.index.astype(str),
        attempt_counts.values,
        color="#3498db",
        edgecolor="white",
    )
    total = attempt_counts.sum()
    for i, (idx, val) in enumerate(attempt_counts.items()):
        axes[1].text(i, val + 0.5, f"{val/total:.1%}", ha="center", fontsize=9)
    axes[1].set_title("Retrieval Attempts Distribution", fontweight="bold")
    axes[1].set_xlabel("Number of Retrieval Attempts")
    axes[1].set_ylabel("Count")

    plt.suptitle("RAG Retrieval Statistics", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "retrieval_statistics.png", dpi=150, bbox_inches="tight")
    plt.show()

## 10. Self-Consistency Analysis

In [ ]:
from eval.metrics.self_consistency_metrics import sc_agreement_rate, sc_by_feature

if "verification_method" in df.columns:
    sc_stats = sc_agreement_rate(df)
    print("Self-Consistency Stats:")
    for k, v in sc_stats.items():
        print(f"  {k}: {v}")

    sc_feat = sc_by_feature(df)

    if not sc_feat.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        x = range(len(sc_feat))
        ax.bar(
            x,
            sc_feat["sc_passed"],
            label="Passed",
            color="#2ecc71",
            edgecolor="white",
        )
        ax.bar(
            x,
            sc_feat["sc_failed"],
            bottom=sc_feat["sc_passed"],
            label="Failed",
            color="#e74c3c",
            edgecolor="white",
        )
        ax.set_xticks(list(x))
        ax.set_xticklabels(
            sc_feat["feature_name"].str.replace("feature_", "").str.replace("_", " "),
            rotation=15,
        )
        ax.set_title("Self-Consistency Results by High-Risk Feature", fontweight="bold")
        ax.set_ylabel("Count")
        ax.legend()
        plt.tight_layout()
        plt.savefig(REPORTS_DIR / "self_consistency_results.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("No self-consistency runs found in dataset.")

## 11. HCAT Safety Score — All Cases

In [ ]:
from eval.metrics.hcat_metrics import compute_batch_hcat, hcat_report_to_df

if "case_id" in df.columns and "verdict" in df.columns:
    run_id_for_report = df.get("run_id", pd.Series(["demo_run"])).iloc[0]
    prompt_id_for_report = df.get("prompt_id", pd.Series(["unknown"])).iloc[0]
    model_id_for_report = df.get("model_id", pd.Series(["unknown"])).iloc[0]

    report = compute_batch_hcat(
        df,
        run_id=str(run_id_for_report),
        prompt_id=str(prompt_id_for_report),
        model_id=str(model_id_for_report),
    )

    hcat_df = hcat_report_to_df(report)
    print(f"HCAT Report — {len(hcat_df)} cases")
    print(f"  Mean fabrication rate: {report.mean_fabrication_rate:.1%}")
    print(f"  Mean accuracy:         {report.mean_accuracy:.1%}")
    print(f"  Mean omission rate:    {report.mean_omission_rate:.1%}")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, col, label, color in zip(
        axes,
        ["fabrication_rate", "accuracy", "omission_rate"],
        ["Fabrication Rate", "Accuracy", "Omission Rate"],
        ["#e74c3c", "#2ecc71", "#f39c12"],
    ):
        vals = hcat_df[col].dropna() * 100
        ax.hist(vals, bins=10, color=color, edgecolor="white", alpha=0.85)
        ax.axvline(vals.mean(), color="black", linestyle="--",
                   label=f"Mean={vals.mean():.1f}%")
        ax.set_title(f"{label} Distribution", fontweight="bold")
        ax.set_xlabel("%")
        ax.set_ylabel("Cases")
        ax.legend(fontsize=9)

    plt.suptitle("HCAT Safety Metrics — Batch Report", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "hcat_safety_metrics.png", dpi=150, bbox_inches="tight")
    plt.show()

    hcat_df.describe()

## 12. Export All Figures Summary

In [ ]:
saved_pngs = sorted(REPORTS_DIR.glob("*.png"))
print(f"Figures saved to: {REPORTS_DIR}")
print(f"Total figures: {len(saved_pngs)}")
for p in saved_pngs:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<55} {size_kb:6.1f} KB")